In [1]:
import numpy as np #arrrays
import pandas as pd #data set structure
import matplotlib.pyplot as plt #visualisation data strructre
import yfinance as yf #package

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

from itertools import combinations

from sklearn.feature_selection import mutual_info_regression


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TICKER = "AAPL"
df = yf.download(TICKER, '2020-01-01')


[*********************100%***********************]  1 of 1 completed


In [2]:
class PredictionModel(nn.Module):
    
        def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
            super(PredictionModel, self).__init__()
    
            self.num_layers = num_layers
            self.hidden_dim = hidden_dim
    
            self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first = True, dropout=0.2)
            self.fc = nn.Linear(hidden_dim, output_dim)
        def forward(self, x):
            h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device = device)
            c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device = device)
    
            out, (hn, cn) = self.lstm(x, (h0.detach(), c0.detach()))
            out = self.fc(out[:, -1, :])
            return out

In [3]:

def run_experiment(df, TICKER, SEQ_LENGTH, TOP_FEATURES, HIDDEN_DIM, NUM_EPOCHS, TRAIN_SPLIT, NUM_LAYERS, DROPOUT, BATCH_SIZE, LEARNING_RATE):
    df["Returns"] = df["Close"].pct_change()
    
    df["SMA_10"] = df["Close"].rolling(10).mean()
    df["SMA_50"] = df["Close"].rolling(50).mean()
    
    df["EMA_10"] = df["Close"].ewm(span=10, adjust=False).mean()
    df["EMA_50"] = df["Close"].ewm(span=50, adjust=False).mean()
    
    df["Volatility"] = df["Returns"].rolling(20).std()
    
    delta = df["Close"].diff()
    
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(14).mean()
    avg_loss = loss.rolling(14).mean()
    
    rs = avg_gain / avg_loss
    df["RSI"] = 100 - (100 / (1 + rs))
    
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    
    df["MACD"] = ema12 - ema26
    
    high_low = df["High"] - df["Low"]
    high_close = np.abs(df["High"] - df["Close"].shift())
    low_close = np.abs(df["Low"] - df["Close"].shift())
    
    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)
    
    df["ATR"] = tr.rolling(14).mean()

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    
    # Step 1: create target first
    df["Tomorrow_Return"] = df["Close"].shift(-1) / df["Close"] - 1
    
    
    
    features = [
        "Open", "High", "Low", "Close", "Volume",
        "RSI", "MACD", "EMA_10", "EMA_50",
        "SMA_10", "SMA_50",
        "ATR", "Returns", "Volatility"
    ]
    X = df[features]
        

    data = df[features + ["Tomorrow_Return"]]
    
    # 4. drop NaNs ONCE
    data = data.dropna()
    
    # 5. split cleanly
    X = data[features]
    y = data["Tomorrow_Return"]

    train_size = int(TRAIN_SPLIT * len(data))
    
    train_df = data.iloc[:train_size]
    test_df = data.iloc[train_size:]



    X_train = train_df[features]
    y_train = train_df["Tomorrow_Return"]
    
    X_test = test_df[features]
    y_test = test_df["Tomorrow_Return"]

    mi = mutual_info_regression(X_train, y_train, random_state=42)
    
    mi_scores = pd.Series(mi, index=X_train.columns)
    mi_scores = mi_scores.sort_values(ascending=False)
    
    top_features = mi_scores.head(TOP_FEATURES).index
    
    X = X[top_features]
    
    X_train = X_train[top_features]
    X_test = X_test[top_features]




    feature_scaler = MinMaxScaler()
    
    X_train_scaled = feature_scaler.fit_transform(X_train) #fit trains the scaler, transform scales the training data
    
    X_test_scaled = feature_scaler.transform(X_test)
    
    target_scaler = MinMaxScaler()
    y_train_scaled = target_scaler.fit_transform(y_train.to_frame())
    
    y_test_scaled = target_scaler.transform(y_test.to_frame())


    
    model = PredictionModel(input_dim=TOP_FEATURES, hidden_dim=HIDDEN_DIM, num_layers=NUM_LAYERS, output_dim=1).to(device)
    #hidden dim = long term memory
    criterion = nn.MSELoss() #mean squared error
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

    
    seq_length = SEQ_LENGTH
    
    X_train_seq = []
    y_train_seq = []
    
    for i in range(seq_length, len(X_train_scaled)):
        X_train_seq.append(X_train_scaled[i-seq_length:i])
        y_train_seq.append(y_train_scaled[i])
    
    #X_train = np.array(X_train)
    #y_train = np.array(y_train)
    
    # convert to tensors
    X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_seq, dtype=torch.float32)
    
    X_test_seq = []
    y_test_seq = []
    
    for i in range(seq_length, len(X_test_scaled)):
        X_test_seq.append(X_test_scaled[i-seq_length:i])
        y_test_seq.append(y_test_scaled[i])
    
    X_test_tensor = torch.tensor(np.array(X_test_seq), dtype=torch.float32)
    y_test_tensor = torch.tensor(np.array(y_test_seq), dtype=torch.float32)



    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False) 


    num_epochs = NUM_EPOCHS
    
    X_train_tensor = X_train_tensor.to(device)
    y_train_tensor = y_train_tensor.to(device)
    X_test_tensor = X_test_tensor.to(device)
    y_test_tensor = y_test_tensor.to(device)
    
    
    for epoch in range(num_epochs):
    
        model.train()
    
        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            
            y_pred = model(X_batch)
            loss = criterion(y_pred, y_batch)
    
            optimizer.zero_grad()
    
            loss.backward() #weights different qualities based on how they change the loss
            #print(model.fc.weight.grad)
    
            optimizer.step()
    
    
    model.eval()
    
    with torch.no_grad():
    
        y_train_pred = model(X_train_tensor)
        y_test_pred = model(X_test_tensor)

    
    y_train_pred_tensor = y_train_pred
    y_test_pred_tensor = y_test_pred
    
    y_train_pred = target_scaler.inverse_transform(y_train_pred_tensor.cpu().numpy()) #taking scaled data back to original price space
    y_train = target_scaler.inverse_transform(y_train_tensor)
    y_test_pred = target_scaler.inverse_transform(y_test_pred_tensor.cpu().numpy())
    y_test = target_scaler.inverse_transform(y_test_tensor)


    train_rmse = root_mean_squared_error(y_train[:, 0], y_train_pred[:, 0])
    test_rmse = root_mean_squared_error(y_test[:, 0], y_test_pred[:, 0])


    baseline_pred = np.zeros_like(y_test)
    
    baseline_rmse = root_mean_squared_error(y_test[:, 0], baseline_pred[:, 0])
    model_rmse = root_mean_squared_error(y_test[:, 0], y_test_pred[:, 0])
    

    naive = test_df["Returns"].iloc[SEQ_LENGTH:].to_numpy()
    
    direction = np.mean(
        np.sign(naive) == np.sign(y_test[:,0])
    )
    

    return train_rmse, test_rmse, direction, y_test_pred, y_test
        

In [4]:
results = []

for seq_length in [5, 10, 20, 30, 60]:
    train_rmse, test_rmse, direction, y_test_pred, y_test = run_experiment(df, TICKER, SEQ_LENGTH = seq_length, TOP_FEATURES=10, HIDDEN_DIM = 64, NUM_EPOCHS = 100, TRAIN_SPLIT = 0.8, NUM_LAYERS = 2, DROPOUT = 0.2 , BATCH_SIZE = 32, LEARNING_RATE = 0.001
    )

    results.append({
        "Sequence Length": seq_length,
        "Train RMSE": train_rmse,
        "Test RMSE": test_rmse,
        "Direction": direction,
        "Mean return prediction:": np.mean(y_test_pred),
        "Mean actual return:": np.mean(y_test),
        "Model std": np.std(y_test_pred),
        "Actual std": np.std(y_test)
    })

results = pd.DataFrame(results)
print(results)

C:\Users\lilyr\AppData\Local\Temp\ipykernel_14392\3408523353.py:122: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  X_train_tensor = torch.tensor(X_train_seq, dtype=torch.float32)


   Sequence Length  Train RMSE  Test RMSE  Direction  Mean return prediction:  \
0                5    0.018563   0.018903   0.522436                 0.000188   
1               10    0.018214   0.015938   0.521173                -0.000030   
2               20    0.017943   0.015329   0.521886                 0.001060   
3               30    0.018038   0.022511   0.512195                -0.010281   
4               60    0.018486   0.015462   0.513619                -0.001103   

   Mean actual return:  Model std  Actual std  
0             0.001532   0.002782    0.018845  
1             0.001715   0.003170    0.015754  
2             0.001444   0.000659    0.015368  
3             0.001674   0.012620    0.015311  
4             0.001813   0.003369    0.015112  
